In [1]:
from pathlib import Path

import pandas as pd

DATA_DIR = Path("C:/Users/Jabulani.Mcineka/workspace/Data-QC/Mental _health_dat_QC")
RAW_ACTIVITY_PATH = DATA_DIR / "22763_AUDITC.csv"
CURATED_AUDITC_PATH = DATA_DIR / "Audit c Dataset.csv"

raw_df1 = pd.read_csv(RAW_ACTIVITY_PATH)
raw_df2 = pd.read_csv(CURATED_AUDITC_PATH)

df1_clean = raw_df1.rename(columns={
    "Participant ID": "participant_id_source",
    "Session Scheduled Time": "scheduled_time_source",
    "[2_SAQ] Question 2 of Survey 22763": "question2_response_source",
    "[3_SAQ] Question 3 of Survey 22763": "question3_response_source",
    "[4_SAQ] Question 4 of Survey 22763": "question4_response_source",
})
df2_clean = raw_df2.rename(columns={
    "IIntID": "iint_id_source",
    "Map_ID": "map_id_source",
    "sched_time": "scheduled_time_source",
    "saq2_22763": "question2_response_source",
    "saq3_22763": "question3_response_source",
    "saq4_22763": "question4_response_source",
})

assert len(df1_clean) == len(raw_df1)
assert len(df2_clean) == len(raw_df2)
{"big_dataset_rows": len(df1_clean), "mini_dataset_rows": len(df2_clean)}

{'big_dataset_rows': 584, 'mini_dataset_rows': 583}

In [6]:
question_columns = ["question2", "question3", "question4"]
question3_mini_map = {"2-Jan": "1 - 2", "4-Mar": "3 - 4", "6-May": "5 - 6", "9-Jul": "7 - 9"}

def normalize_answer(series):
    return series.astype("string").str.strip().replace(".", pd.NA)

def normalize_prefixed_answer(series):
    return normalize_answer(series).str.split(" - ", n=1).str[-1].astype("string")

raw_answers_1 = df1_clean[[f"{name}_response_source" for name in question_columns]].apply(normalize_prefixed_answer)
mini_answers = df2_clean[[f"{name}_response_source" for name in question_columns]].apply(normalize_answer)
mini_answers["question2_response_source"] = normalize_prefixed_answer(mini_answers["question2_response_source"])
mini_answers["question3_response_source"] = normalize_prefixed_answer(
    mini_answers["question3_response_source"].replace(question3_mini_map)
)

# Question 4 already uses the same text labels in both datasets.
df1_standard = pd.DataFrame({
    "big_row_id": raw_df1.index,
    "participant_id": df1_clean["participant_id_source"].astype("string").str.strip(),
    "scheduled_date": pd.to_datetime(df1_clean["scheduled_time_source"], errors="coerce", utc=True).dt.date,
})
for name in question_columns:
    df1_standard[name] = raw_answers_1[f"{name}_response_source"]

df2_standard = pd.DataFrame({
    "mini_row_id": raw_df2.index,
    "iint_id": df2_clean["iint_id_source"].astype("string").str.strip(),
    "map_id_source": df2_clean["map_id_source"].astype("string").str.strip(),
    "scheduled_date": pd.to_datetime(df2_clean["scheduled_time_source"], format="%d-%b-%y", errors="coerce").dt.date,
})
for name in question_columns:
    df2_standard[name] = mini_answers[f"{name}_response_source"]

df2_standard["map_id_digits"] = df2_standard["map_id_source"].str.replace(r"[^0-9]", "", regex=True)
df2_standard["iint_id_prefix_match"] = [map_digits.startswith(iint_id) for map_digits, iint_id in zip(df2_standard["map_id_digits"], df2_standard["iint_id"])]
df2_standard["map_id_suffix"] = [map_digits[len(iint_id):] if match else "" for map_digits, iint_id, match in zip(df2_standard["map_id_digits"], df2_standard["iint_id"], df2_standard["iint_id_prefix_match"])]
df2_standard["participant_id_sequences"] = [tuple(suffix[position:position + 5] for position in range(0, len(suffix), 5)) if suffix and len(suffix) % 5 == 0 else () for suffix in df2_standard["map_id_suffix"]]
df2_standard["participant_id_count"] = df2_standard["participant_id_sequences"].str.len()
df2_standard["map_id_match"] = [prefix_match and bool(sequence) for prefix_match, sequence in zip(df2_standard["iint_id_prefix_match"], df2_standard["participant_id_sequences"]) ]
df2_standard["mapping_ambiguous"] = df2_standard["participant_id_count"] > 1

assert len(df1_standard) == len(raw_df1)
assert len(df2_standard) == len(raw_df2)
df1_standard.head(), df2_standard.head()

(   big_row_id participant_id scheduled_date          question2 question3  \
 0           0         100921     2024-10-03  4 times per month         9   
 1           1         100295     2024-10-03               <NA>      <NA>   
 2           2          43781     2024-10-03  4 times per month         4   
 3           3          89406     2024-10-03    Monthly or less         2   
 4           4          46493     2024-10-03  4 times per month         2   
 
                question4  
 0                 Weekly  
 1                   <NA>  
 2      Less than monthly  
 3                  Never  
 4  Daily or almost daily  ,
    mini_row_id iint_id map_id_source scheduled_date          question2  \
 0            0    2221   222,143,781     2024-10-03  4 times per month   
 1            1    2221   222,143,781     2024-11-03    Monthly or less   
 2            2    2221   222,143,781     2024-12-03  4 times per month   
 3            3    2221   222,143,781     2025-01-03  4 times per m

In [7]:
df2_for_match = df2_standard.copy()
df2_for_match["participant_id"] = df2_for_match["participant_id_sequences"].apply(list)
df2_for_match = df2_for_match.explode("participant_id", ignore_index=True)
df2_for_match["participant_id"] = df2_for_match["participant_id"].astype("string")
df1_standard["participant_id"] = df1_standard["participant_id"].astype("string")
comparison = df2_for_match.merge(df1_standard, on=["participant_id", "scheduled_date"], how="left", suffixes=("_mini", "_big"), indicator=True)
comparison["found_in_big"] = comparison["_merge"].eq("both")
for name in question_columns:
    comparison[f"{name}_match"] = comparison[f"{name}_mini"].eq(comparison[f"{name}_big"])
comparison["record_match"] = comparison["found_in_big"] & comparison[[f"{name}_match" for name in question_columns]].all(axis=1)
comparison.head()

,mini_row_id,iint_id,map_id_source,scheduled_date,question2_mini,question3_mini,question4_mini,map_id_digits,iint_id_prefix_match,map_id_suffix,...,big_row_id,question2_big,question3_big,question4_big,_merge,found_in_big,question2_match,question3_match,question4_match,record_match
0,0,2221,"222,143,781",2024-10-03,4 times per month,4,Less than monthly,222143781,True,43781,...,2.0,4 times per month,4,Less than monthly,both,True,True,True,True,True
1,1,2221,"222,143,781",2024-11-03,Monthly or less,6,Less than monthly,222143781,True,43781,...,92.0,Monthly or less,6,Less than monthly,both,True,True,True,True,True
2,2,2221,"222,143,781",2024-12-03,4 times per month,4,Monthly,222143781,True,43781,...,186.0,4 times per month,4,Monthly,both,True,True,True,True,True
3,3,2221,"222,143,781",2025-01-03,4 times per month,4,Less than monthly,222143781,True,43781,...,283.0,4 times per month,4,Less than monthly,both,True,True,True,True,True
4,4,2221,"222,143,781",2025-02-03,4 times per month,4,Less than monthly,222143781,True,43781,...,385.0,4 times per month,4,Less than monthly,both,True,True,True,True,True


In [8]:
mini_row_status = comparison.groupby("mini_row_id", as_index=False)["found_in_big"].any().rename(columns={"found_in_big": "mini_row_found_in_big"})
mini_row_status["mini_row_status"] = mini_row_status["mini_row_found_in_big"].map({True: "found_in_big", False: "missing_from_big"})
comparison = comparison.merge(mini_row_status, on="mini_row_id", how="left")
mini_coverage = df2_standard.merge(mini_row_status, on="mini_row_id", how="left")
assert len(mini_coverage) == len(raw_df2)
comparison_summary = {
    "big_dataset_rows_preserved": len(df1_standard),
    "mini_dataset_rows_preserved": len(df2_standard),
    "mini_rows_found_in_big": int(mini_row_status["mini_row_found_in_big"].sum()),
    "mini_rows_missing_from_big": int((~mini_row_status["mini_row_found_in_big"]).sum()),
    "matched_participant_rows": int(comparison["found_in_big"].sum()),
    "fully_matching_participant_rows": int(comparison["record_match"].sum()),
    "multi_participant_map_ids": int((df2_standard["participant_id_count"] > 1).sum()),
}
mini_rows_found_in_big = mini_coverage.loc[mini_coverage["mini_row_found_in_big"]].reset_index(drop=True)
mini_rows_missing_from_big = mini_coverage.loc[~mini_coverage["mini_row_found_in_big"]].reset_index(drop=True)
fully_matching_rows = comparison.loc[comparison["record_match"]].reset_index(drop=True)
comparison_summary

{'big_dataset_rows_preserved': 584,
 'mini_dataset_rows_preserved': 583,
 'mini_rows_found_in_big': 577,
 'mini_rows_missing_from_big': 6,
 'matched_participant_rows': 577,
 'fully_matching_participant_rows': 577,
 'multi_participant_map_ids': 72}

In [10]:
print(f"Total mini Audit C dataset rows: {comparison_summary['mini_dataset_rows_preserved']}")
display(df2_standard.head())
print(f"Total mini Audit C rows found in big dataset: {comparison_summary['mini_rows_found_in_big']}")
display(mini_rows_found_in_big.head())
print(f"Total mini Audit C rows missing from big dataset: {comparison_summary['mini_rows_missing_from_big']}")
display(mini_rows_missing_from_big.head(6))
print(f"Total fully matching Audit C participant rows: {comparison_summary['fully_matching_participant_rows']}")
display(fully_matching_rows.head())

Total mini Audit C dataset rows: 583


,mini_row_id,iint_id,map_id_source,scheduled_date,question2,question3,question4,map_id_digits,iint_id_prefix_match,map_id_suffix,participant_id_sequences,participant_id_count,map_id_match,mapping_ambiguous
0,0,2221,"222,143,781",2024-10-03,4 times per month,4,Less than monthly,222143781,True,43781,"(43781,)",1,True,False
1,1,2221,"222,143,781",2024-11-03,Monthly or less,6,Less than monthly,222143781,True,43781,"(43781,)",1,True,False
2,2,2221,"222,143,781",2024-12-03,4 times per month,4,Monthly,222143781,True,43781,"(43781,)",1,True,False
3,3,2221,"222,143,781",2025-01-03,4 times per month,4,Less than monthly,222143781,True,43781,"(43781,)",1,True,False
4,4,2221,"222,143,781",2025-02-03,4 times per month,4,Less than monthly,222143781,True,43781,"(43781,)",1,True,False


Total mini Audit C rows found in big dataset: 577


,mini_row_id,iint_id,map_id_source,scheduled_date,question2,question3,question4,map_id_digits,iint_id_prefix_match,map_id_suffix,participant_id_sequences,participant_id_count,map_id_match,mapping_ambiguous,mini_row_found_in_big,mini_row_status
0,0,2221,"222,143,781",2024-10-03,4 times per month,4,Less than monthly,222143781,True,43781,"(43781,)",1,True,False,True,found_in_big
1,1,2221,"222,143,781",2024-11-03,Monthly or less,6,Less than monthly,222143781,True,43781,"(43781,)",1,True,False,True,found_in_big
2,2,2221,"222,143,781",2024-12-03,4 times per month,4,Monthly,222143781,True,43781,"(43781,)",1,True,False,True,found_in_big
3,3,2221,"222,143,781",2025-01-03,4 times per month,4,Less than monthly,222143781,True,43781,"(43781,)",1,True,False,True,found_in_big
4,4,2221,"222,143,781",2025-02-03,4 times per month,4,Less than monthly,222143781,True,43781,"(43781,)",1,True,False,True,found_in_big


Total mini Audit C rows missing from big dataset: 6


,mini_row_id,iint_id,map_id_source,scheduled_date,question2,question3,question4,map_id_digits,iint_id_prefix_match,map_id_suffix,participant_id_sequences,participant_id_count,map_id_match,mapping_ambiguous,mini_row_found_in_big,mini_row_status
0,483,130789,"1,307,895,895,089,590",2024-10-03,Never,2,Never,1307895895089590,True,5895089590,"(58950, 89590)",2,True,True,False,missing_from_big
1,484,130789,"1,307,895,895,089,590",2024-11-03,Never,2,Never,1307895895089590,True,5895089590,"(58950, 89590)",2,True,True,False,missing_from_big
2,485,130789,"1,307,895,895,089,590",2024-12-03,Never,2,Never,1307895895089590,True,5895089590,"(58950, 89590)",2,True,True,False,missing_from_big
3,486,130789,"1,307,895,895,089,590",2025-01-03,Never,2,Never,1307895895089590,True,5895089590,"(58950, 89590)",2,True,True,False,missing_from_big
4,487,130789,"1,307,895,895,089,590",2025-02-03,Never,2,Never,1307895895089590,True,5895089590,"(58950, 89590)",2,True,True,False,missing_from_big
5,488,130789,"1,307,895,895,089,590",2025-03-03,Never,2,Never,1307895895089590,True,5895089590,"(58950, 89590)",2,True,True,False,missing_from_big


Total fully matching Audit C participant rows: 577


,mini_row_id,iint_id,map_id_source,scheduled_date,question2_mini,question3_mini,question4_mini,map_id_digits,iint_id_prefix_match,map_id_suffix,...,question3_big,question4_big,_merge,found_in_big,question2_match,question3_match,question4_match,record_match,mini_row_found_in_big,mini_row_status
0,0,2221,"222,143,781",2024-10-03,4 times per month,4,Less than monthly,222143781,True,43781,...,4,Less than monthly,both,True,True,True,True,True,True,found_in_big
1,1,2221,"222,143,781",2024-11-03,Monthly or less,6,Less than monthly,222143781,True,43781,...,6,Less than monthly,both,True,True,True,True,True,True,found_in_big
2,2,2221,"222,143,781",2024-12-03,4 times per month,4,Monthly,222143781,True,43781,...,4,Monthly,both,True,True,True,True,True,True,found_in_big
3,3,2221,"222,143,781",2025-01-03,4 times per month,4,Less than monthly,222143781,True,43781,...,4,Less than monthly,both,True,True,True,True,True,True,found_in_big
4,4,2221,"222,143,781",2025-02-03,4 times per month,4,Less than monthly,222143781,True,43781,...,4,Less than monthly,both,True,True,True,True,True,True,found_in_big
